# Kimi

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HOME"]     = "/root/autodl-tmp/LLM_Model"

import json
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import snapshot_download
from kimia_infer.api.kimia import KimiAudio

CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "moonshotai/Kimi-Audio-7B-Instruct"

LOCAL_MODEL_PATH = snapshot_download(MODEL_ID, cache_dir=CACHE_DIR)

Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

In [2]:
model = KimiAudio(model_path=LOCAL_MODEL_PATH, load_detokenizer=True)
print("Kimi Audio model loaded.")

2026-04-01 02:52:11.253 | INFO     | kimia_infer.api.kimia:__init__:16 - Loading kimi-audio main model
2026-04-01 02:52:11.256 | INFO     | kimia_infer.api.kimia:__init__:25 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-04-01 02:52:11.257 | INFO     | kimia_infer.api.kimia:__init__:26 - Loading whisper model
`torch_dtype` is deprecated! Use `dtype` instead!
using normal flash attention


Loading checkpoint shards:   0%|          | 0/36 [00:00<?, ?it/s]

2026-04-01 02:52:19.079 | INFO     | kimia_infer.api.prompt_manager:__init__:20 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-04-01 02:52:19.082 | INFO     | kimia_infer.api.prompt_manager:__init__:21 - Loading whisper model
2026-04-01 02:52:20.062 | INFO     | kimia_infer.api.prompt_manager:__init__:30 - Loading text tokenizer
2026-04-01 02:52:20.258 | INFO     | kimia_infer.api.kimia:__init__:41 - Loading detokenizer


ninja: no work to do.


/root/autodl-tmp/envs/kimi/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading '/root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b/vocoder/model.pt'
Complete.
using rope base theta = 10000.0, interpolation factor = 1.0
Currently using bfloat16 for PrefixFlowMatchingDetokenizer
Kimi Audio model loaded.


In [3]:
SYSTEM_PROMPT = (
    "Reply one word: Dementia or Control."
)
USER_PROMPT = "Dementia or Control?"

In [4]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="kimi_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [5]:
VALID_LABELS = {"Dementia", "Healthy"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    messages = [
        {"role": "user", "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT},
        {"role": "user", "message_type": "audio", "content": str(wav_path)},
    ]
    _, text = model.generate(messages, output_type="text")
    return text


def parse_prediction(raw: str) -> str | None:
    """Extract prediction from model output via keyword matching."""
    text = raw.lower()
    has_dementia = "dementia" in text
    has_control  = "control" in text or "healthy" in text
    if has_dementia and not has_control:
        return "Dementia"
    if has_control and not has_dementia:
        return "Control"
    return None

In [6]:
OUTPUT_DIR = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result")


def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            pred = parse_prediction(raw)
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    # Save predictions to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / f"{name}.csv"
    pd.DataFrame(predictions).to_csv(out_csv, index=False)
    print(f"  Saved to {out_csv}")

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [7]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt"
evaluate_dataset(csv, audio_dir, "Pitt-raw")

[Pitt-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt, exists=True


Pitt-raw:   0%|          | 1/551 [00:01<17:26,  1.90s/it]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-raw:   0%|          | 2/551 [00:02<09:14,  1.01s/it]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


...

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-raw:  75%|███████▌  | 415/551 [02:30<01:03,  2.14it/s]

  INVALID [414] session=271-2 true=Dementia raw='A woman is telling another woman about a story involving a child and a cookie jar.'


Pitt-raw: 100%|██████████| 551/551 [03:16<00:00,  2.81it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-raw.csv
[Pitt-raw]
  Accuracy:    0.5764
  F1:          0.7216
  Control Acc: 0.0620
  Dementia Acc:0.9805
  Valid: 550/551  Skipped: 0


In [8]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

[Lu-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Lu, exists=True


Lu-raw:   1%|▏         | 1/74 [00:00<00:26,  2.80it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-raw:   3%|▎         | 2/74 [00:00<00:22,  3.24it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-raw:   4%|▍         | 3/74 [00:00<00:22,  3.22it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-raw: 100%|██████████| 74/74 [00:20<00:00,  3.55it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Lu-raw.csv
[Lu-raw]
  Accuracy:    0.4865
  F1:          0.6415
  Control Acc: 0.0556
  Dementia Acc:0.8947
  Valid: 74/74  Skipped: 0


In [9]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

[Pitt-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Demucs, exists=True


Pitt-Demucs:   0%|          | 1/551 [00:00<03:37,  2.52it/s]

  DEBUG [0] session=002-0 raw='Control.' pred=Control


Pitt-Demucs:   0%|          | 2/551 [00:00<03:29,  2.62it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


...

  INVALID [154] session=175-2 true=Control raw='What I see.'


Pitt-Demucs:  92%|█████████▏| 508/551 [03:02<00:23,  1.82it/s]

  INVALID [507] session=579-0 true=Dementia raw='A woman doing dishes, a boy climbing up to get some cookies, a girl waiting to get some of the cookies, a bench is falling over with the boy, water is dripping out on the floor.'


Pitt-Demucs: 100%|██████████| 551/551 [03:15<00:00,  2.82it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-Demucs.csv
[Pitt-Demucs]
  Accuracy:    0.5738
  F1:          0.7188
  Control Acc: 0.0664
  Dementia Acc:0.9708
  Valid: 549/551  Skipped: 0


In [10]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu-Demucs")

[Lu-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Demucs, exists=True


Lu-Demucs:   1%|▏         | 1/74 [00:00<00:23,  3.08it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Demucs:   3%|▎         | 2/74 [00:00<00:20,  3.48it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-Demucs:   4%|▍         | 3/74 [00:00<00:20,  3.50it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Demucs: 100%|██████████| 74/74 [00:19<00:00,  3.76it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Lu-Demucs.csv
[Lu-Demucs]
  Accuracy:    0.5270
  F1:          0.6847
  Control Acc: 0.0278
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0


In [11]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

[Pitt-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Denoiser, exists=True


Pitt-Denoiser:   0%|          | 1/551 [00:00<02:57,  3.10it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-Denoiser:   0%|          | 2/551 [00:00<02:57,  3.09it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


...

  DEBUG [2] session=002-2 raw='Dementia.' pred=Dementia


Pitt-Denoiser:  62%|██████▏   | 341/551 [01:46<01:52,  1.86it/s]

  INVALID [340] session=164-1 true=Dementia raw='The sink is running, water is on the floor, a boy is standing on a stool that is going to tip over, and the lid is off a cookie jar.'


Pitt-Denoiser: 100%|██████████| 551/551 [02:53<00:00,  3.17it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-Denoiser.csv
[Pitt-Denoiser]
  Accuracy:    0.5691
  F1:          0.7189
  Control Acc: 0.0413
  Dementia Acc:0.9838
  Valid: 550/551  Skipped: 0


In [12]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

[Lu-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Denoiser, exists=True


Lu-Denoiser:   1%|▏         | 1/74 [00:00<00:19,  3.69it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Denoiser:   3%|▎         | 2/74 [00:00<00:18,  3.97it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-Denoiser:   4%|▍         | 3/74 [00:00<00:17,  4.00it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Denoiser: 100%|██████████| 74/74 [00:17<00:00,  4.19it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Lu-Denoiser.csv
[Lu-Denoiser]
  Accuracy:    0.5405
  F1:          0.6909
  Control Acc: 0.0556
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0


In [13]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

[Pitt-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-FRCRN_SE, exists=True


Pitt-FRCRN_SE:   0%|          | 1/551 [00:00<03:04,  2.98it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-FRCRN_SE:   0%|          | 2/551 [00:00<03:09,  2.90it/s]

  DEBUG [1] session=002-1 raw='Dementia.' pred=Dementia


...

  DEBUG [2] session=002-2 raw='Dementia.' pred=Dementia


Pitt-FRCRN_SE:  75%|███████▌  | 415/551 [02:13<00:54,  2.48it/s]

  INVALID [414] session=271-2 true=Dementia raw='The woman is telling a story about a boy and a mother.'


Pitt-FRCRN_SE: 100%|██████████| 551/551 [02:53<00:00,  3.18it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-FRCRN_SE.csv
[Pitt-FRCRN_SE]
  Accuracy:    0.5745
  F1:          0.7201
  Control Acc: 0.0620
  Dementia Acc:0.9773
  Valid: 550/551  Skipped: 0


In [14]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

[Lu-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-FRCRN_SE, exists=True


Lu-FRCRN_SE:   1%|▏         | 1/74 [00:00<00:20,  3.49it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-FRCRN_SE:   3%|▎         | 2/74 [00:00<00:19,  3.69it/s]

  DEBUG [1] session=F22_001 raw='Dementia.' pred=Dementia


...

  INVALID [3] session=F29_000 true=Control raw="I see a lady looking out the window. A lady looking out the window. And I see the kid looking at that picture up there. Sorry, what'd you say? I see the kid looking at this picture. And I see this young girl watching him. Looking up right at him. Watching what he's doing. Watching him looking at the"


Lu-FRCRN_SE:   7%|▋         | 5/74 [00:04<01:06,  1.04it/s]

  INVALID [4] session=F29_001 true=Control raw='Dementia or Control?'


Lu-FRCRN_SE: 100%|██████████| 74/74 [00:20<00:00,  3.60it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Lu-FRCRN_SE.csv
[Lu-FRCRN_SE]
  Accuracy:    0.5417
  F1:          0.6916
  Control Acc: 0.0588
  Dementia Acc:0.9737
  Valid: 72/74  Skipped: 0


In [15]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

[Pitt-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-MossFormer, exists=True


Pitt-MossFormer:   0%|          | 1/551 [00:00<03:06,  2.95it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-MossFormer:   0%|          | 2/551 [00:00<03:09,  2.90it/s]

  DEBUG [1] session=002-1 raw='Dementia.' pred=Dementia


Pitt-MossFormer:   1%|          | 3/551 [00:01<03:04,  2.97it/s]

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-MossFormer: 100%|██████████| 551/551 [02:53<00:00,  3.18it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-MossFormer.csv
[Pitt-MossFormer]
  Accuracy:    0.5808
  F1:          0.7227
  Control Acc: 0.0785
  Dementia Acc:0.9741
  Valid: 551/551  Skipped: 0


In [16]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

[Lu-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-MossFormer, exists=True


Lu-MossFormer:   1%|▏         | 1/74 [00:00<00:21,  3.44it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-MossFormer:   3%|▎         | 2/74 [00:00<00:18,  3.87it/s]

  DEBUG [1] session=F22_001 raw='dementia' pred=Dementia


...

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-MossFormer:   7%|▋         | 5/74 [00:01<00:17,  4.02it/s]

  INVALID [4] session=F29_001 true=Control raw='Dementia or Control?'


Lu-MossFormer: 100%|██████████| 74/74 [00:17<00:00,  4.17it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Lu-MossFormer.csv
[Lu-MossFormer]
  Accuracy:    0.5342
  F1:          0.6792
  Control Acc: 0.0857
  Dementia Acc:0.9474
  Valid: 73/74  Skipped: 0


In [17]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

[Pitt-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Resemble, exists=True


Pitt-Resemble:   0%|          | 1/551 [00:00<03:41,  2.49it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-Resemble:   0%|          | 2/551 [00:00<03:32,  2.58it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


...

  INVALID [355] session=183-0 true=Dementia raw='A girl is playing with a doll.'


Pitt-Resemble:  75%|███████▌  | 415/551 [02:28<00:55,  2.44it/s]

  INVALID [414] session=271-2 true=Dementia raw='The woman is talking about a picture.'


Pitt-Resemble: 100%|██████████| 551/551 [03:13<00:00,  2.84it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-Resemble.csv
[Pitt-Resemble]
  Accuracy:    0.5683
  F1:          0.7148
  Control Acc: 0.0620
  Dementia Acc:0.9674
  Valid: 549/551  Skipped: 0


In [18]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")

[Lu-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Resemble, exists=True


Lu-Resemble:   1%|▏         | 1/74 [00:00<00:22,  3.30it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Resemble:   3%|▎         | 2/74 [00:00<00:19,  3.61it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


...

  INVALID [4] session=F29_001 true=Control raw='Dementia or Control?'


Lu-Resemble:  39%|███▉      | 29/74 [00:07<00:12,  3.70it/s]

  INVALID [28] session=F49_001 true=Control raw='Dementia or Control?'


Lu-Resemble: 100%|██████████| 74/74 [00:19<00:00,  3.71it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Lu-Resemble.csv
[Lu-Resemble]
  Accuracy:    0.5694
  F1:          0.7103
  Control Acc: 0.0882
  Dementia Acc:1.0000
  Valid: 72/74  Skipped: 0
